# 3. Spherical converging shock

## Physical picture

This benchmark starts with uniform gas moving inward in spherical geometry. Every spherical shell is moving toward smaller radius, so the same flow is squeezed into progressively smaller shell volumes. The converging flow focuses mass and kinetic energy toward the inner reflecting boundary. A strong compression wave develops, and kinetic energy is converted into thermal energy at the resulting shock.

The maintained configuration uses an inner radius of `0.01 cm`, an outer radius of `1 cm`, density `1 g cm^-3`, temperature `1e-10 K`, and inward velocity `u = -1 cm s^-1`. There is no gravity or external heating: the interesting behavior comes from spherical geometry, the reflecting boundaries, and the Euler equations. The notebook uses a reduced 128-cell mesh so it is practical in a hosted Colab runtime.

## Spherical Euler equations

For radial flow with no angular dependence, the conservation laws are

$$\frac{\partial \rho}{\partial t}+\frac{1}{r^2}\frac{\partial}{\partial r}(r^2\rho u)=0,$$

$$\frac{\partial (\rho u)}{\partial t}+\frac{1}{r^2}\frac{\partial}{\partial r}\left[r^2(\rho u^2+p)\right]=\frac{2p}{r},$$

$$\frac{\partial E}{\partial t}+\frac{1}{r^2}\frac{\partial}{\partial r}\left[r^2u(E+p)\right]=0,$$

where `E` is total energy density and

$$E=\frac{p}{\gamma-1}+\frac{1}{2}\rho u^2.$$

The factors of `r^2` are the geometric area of a spherical surface. The radial momentum equation also contains the pressure-geometry term `2p/r`. These terms are the essential difference from a Cartesian shock tube: even a spatially uniform radial state can evolve because the area and volume of the shells change with radius.

## Shell-integrated finite-volume form

A spherical finite-volume cell is a shell between radii `r_{i-1/2}` and `r_{i+1/2}`. Its volume is

$$V_i=\frac{4\pi}{3}\left(r_{i+1/2}^3-r_{i-1/2}^3\right).$$

Instead of treating `1/r^2` as an ordinary Cartesian derivative, the solver updates shell-integrated conserved quantities using the physical face areas `4 pi r^2`. For example, the mass update has the form

$$\frac{d}{dt}(\rho_i V_i)=-4\pi\left[r_{i+1/2}^2(\rho u)_{i+1/2}-r_{i-1/2}^2(\rho u)_{i-1/2}\right].$$

The same face-flux construction is used for momentum and energy, together with the spherical pressure source. At the inner and outer boundaries, the reflecting condition reverses the normal velocity and prevents mass from crossing the domain.

## What the shock does

Before the shock forms, the gas has substantial inward kinetic energy and very little thermal energy because the initial temperature is extremely small. As shells converge, the flow becomes compressed. Across an adiabatic shock, the Rankine–Hugoniot jump conditions conserve mass, momentum, and total energy, but entropy increases and part of the organized kinetic energy becomes internal energy. The temperature therefore rises sharply near the converging shock.

The runner checks these ideas directly. It verifies that the total mass and total energy remain nearly constant, that the integrated thermal energy increases, and that the maximum temperature grows by at least a factor of five. The plotted profiles show how density and temperature change through the first 13 saved snapshots, which keeps the reference figure focused on the resolved shock-formation phase.

## Why dual energy is enabled

In a high-Mach flow, total energy is dominated by kinetic energy. Computing thermal energy by subtracting kinetic energy from total energy can then lose precision: two large nearly equal numbers are subtracted to obtain a much smaller number. RadHydropy's dual-energy switch evolves an additional internal-energy estimate and chooses a reliable pressure source when the conservative estimate is not sufficiently accurate.

This is important here because pressure determines the shock response and temperature. The dual-energy method should improve pressure recovery without changing the conservative mass and total-energy update. The benchmark therefore tests both geometric shock evolution and the numerical treatment of small thermal energy in a high-Mach converging flow.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.5.3', 'scipy==1.18.1', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
PINNED_MARKER = Path('/content/.radhydropy_dependencies_pinned')
if not PINNED_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
    PINNED_MARKER.touch()
    print('Dependencies installed. Restarting the Colab runtime once to load the pinned NumPy/SciPy binaries.')
    os.kill(os.getpid(), 9)
sys.path.insert(0, str(REPO))
%matplotlib inline
import yaml
from IPython.display import Image, display
from example.SphericalConvergingShock1D import spherical_converging_shock1d

In [ ]:
source_yaml = REPO / 'example' / 'SphericalConvergingShock1D' / 'spherical_converging_shock1d.yaml'
run_dir = Path('/content/radhydropy-spherical-shock')
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir()
config = yaml.safe_load(source_yaml.read_text())
config['initial_condition']['grid_cells'] = 128
config['par']['mesh']['grid_cells'] = 128
config['par']['output']['directory'] = str(run_dir)
config['par']['simulation']['initial_condition_filename'] = 'InitialCondition.hdf5'
config_file = run_dir / 'spherical_shock.yaml'
config_file.write_text(yaml.safe_dump(config))
figure = spherical_converging_shock1d.run(config_file)
print('Completed:', figure)

In [ ]:
display(Image(filename=str(run_dir / 'SphericalConvergingShock1D.jpg')))

## Questions to explore

- Why does the temperature rise near the centre?
- Which part of the plot demonstrates geometric convergence?
- What changes when the resolution is increased to 256 cells?

In [ ]:
# Optional: save the spherical-shock run to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/03_spherical_shock')
# save_dir.mkdir(parents=True, exist_ok=True)
# for file in run_dir.glob('*'):
#     shutil.copy2(file, save_dir / file.name)
# print('Saved to:', save_dir)